In [6]:
import pandas as pd
import matplotlib.pyplot as plt


data = pd.read_csv('data/Student_data.csv')

data.isna().sum()

# pas de Na

# get type for every column
print(data.dtypes)

Student_ID              object
Gender                  object
Age                      int64
Major                   object
Attendance_Pct         float64
Study_Hours_Per_Day    float64
Previous_GPA           float64
Sleep_Hours            float64
Social_Hours_Week        int64
Final_CGPA             float64
dtype: object


In [7]:
import json
from sklearn.model_selection import train_test_split

# Config
DATASET_NAME = "Student_data.csv"
TARGET = "Final_CGPA"
RANDOM_STATE = 42
TEST_SIZE = 0.15
VALID_SIZE = 0.15 / (1 - TEST_SIZE)  # ~0.1765 of remaining to get 15% overall

# Split features / target
X = data.drop(columns=[TARGET])
y = data[TARGET]

# Bin target into 5 quantile-based strata (used ONLY for stratification, not saved)
strata = pd.qcut(y, q=5, labels=False, duplicates="drop")

# 70 / 15 / 15
X_train, X_test, y_train, y_test, s_train, _ = train_test_split(
    X, y, strata, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=strata
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=VALID_SIZE, random_state=RANDOM_STATE, stratify=s_train
)

# Rebuild full DataFrames
data_train = X_train.copy(); data_train[TARGET] = y_train
data_valid = X_valid.copy(); data_valid[TARGET] = y_valid
data_test  = X_test.copy();  data_test[TARGET]  = y_test

# Save splits
data_train.to_csv("data/data_train.csv", index=False)
data_valid.to_csv("data/data_valid.csv", index=False)
data_test.to_csv("data/data_test.csv",  index=False)

print(f"Train : {data_train.shape}, Valid : {data_valid.shape}, Test : {data_test.shape}")

# --- Traçabilité ---
run_info = {
    "dataset":       DATASET_NAME,
    "shape":         {"rows": int(data.shape[0]), "columns": int(data.shape[1])},
    "target":        TARGET,
    "split_fractions": {"train": 0.70, "valid": 0.15, "test": 0.15},
    "random_state":  RANDOM_STATE,
    "stratify":      "quantile bins (q=5) on target — bins used only for splitting",
    "split_sizes":   {
        "train": int(len(data_train)),
        "valid": int(len(data_valid)),
        "test":  int(len(data_test)),
    },
}
with open("data/run_info.json", "w") as f:
    json.dump(run_info, f, indent=4)

print("run_info.json saved.")



Train : (3500, 10), Valid : (750, 10), Test : (750, 10)
run_info.json saved.


In [8]:
import numpy as np
num_col = data.select_dtypes(include=np.number).columns.tolist()
num_col.remove("Final_CGPA")  # remove target
cat_col = data.select_dtypes(exclude=np.number).columns.tolist()

print(f'num_col {num_col}')
print(f'cat_col {cat_col}')

num_col ['Age', 'Attendance_Pct', 'Study_Hours_Per_Day', 'Previous_GPA', 'Sleep_Hours', 'Social_Hours_Week']
cat_col ['Student_ID', 'Gender', 'Major']


In [9]:
from ydata_profiling import ProfileReport

In [13]:
from ydata_profiling import ProfileReport

profile = ProfileReport(data, title="Student Data Profiling Report", explorative=True)
print(profile)
profile.to_file('rapport_analyse_exploratoire.html')

print(open('rapport_analyse_exploratoire.html'))


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 622.69it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

<_io.TextIOWrapper name='rapport_analyse_exploratoire.html' mode='r' encoding='UTF-8'>
